In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter






In [2]:
def helmholtz5(m: np.ndarray,
                    omega: float,
                    dx: float,
                    dz: float,
                    npml: int,
                    vmax: float) -> sp.csr_matrix:
    """
    Build 2D Helmholtz matrix with quadratic PML.

    Parameters
    ----------
    m : (nz, nx) array
        Squared slowness model (1/v^2)
    omega : float
        Angular frequency
    dx, dz : float
        Grid spacing
    npml : int
        Number of PML grid points
    vmax : float
        Maximum velocity (for sigma_max)

    Returns
    -------
    A : sparse CSR matrix (n x n)
    """

    nz, nx = m.shape
    n = nx * nz

    # ------------------------------------------------------
    # Index mapping
    # ------------------------------------------------------
    def idx(i, j):
        return i * nx + j

    # ------------------------------------------------------
    # Construct PML profiles
    # ------------------------------------------------------
    def pml_sigma(npts, d):
        sigma = np.zeros(npts)
        L = npml * d
        sigma_max = 3 * vmax * np.log(100) / (2 * L)

        for i in range(npml):
            x = (npml - i) / npml
            val = sigma_max * x**2
            sigma[i] = val
            sigma[npts - 1 - i] = val

        return sigma

    sigma_x = pml_sigma(nx, dx)
    sigma_z = pml_sigma(nz, dz)

    # ------------------------------------------------------
    # Assemble matrix
    # ------------------------------------------------------
    rows, cols, data = [], [], []

    for i in range(nz):
        for j in range(nx):

            k = idx(i, j)

            sx = 1.0 + 1j * sigma_x[j] / omega
            sz = 1.0 + 1j * sigma_z[i] / omega

            ax = 1.0 / (sx * dx**2)
            az = 1.0 / (sz * dz**2)

            main = -2.0 * ax - 2.0 * az + omega**2 * m[i, j]

            # center
            rows.append(k); cols.append(k); data.append(main)

            # left
            if j > 0:
                rows.append(k); cols.append(idx(i, j-1)); data.append(ax)

            # right
            if j < nx-1:
                rows.append(k); cols.append(idx(i, j+1)); data.append(ax)

            # up
            if i > 0:
                rows.append(k); cols.append(idx(i-1, j)); data.append(az)

            # down
            if i < nz-1:
                rows.append(k); cols.append(idx(i+1, j)); data.append(az)

    A = sp.coo_matrix((data, (rows, cols)), shape=(n, n))
    return A.tocsr()

In [3]:
def helmholtz9(m: np.ndarray,
               omega: float,
               dx: float,
               dz: float,
               npml: int,
               vmax: float) -> sp.csr_matrix:
    """
    2D Helmholtz with quadratic PML
    using correct 9-point 4th-order stencil.
    """

    nz, nx = m.shape
    n = nx * nz

    def idx(i, j):
        return i * nx + j

    # ---------------- PML ----------------
    def pml_sigma(npts, d):
        sigma = np.zeros(npts)
        L = npml * d
        sigma_max = 3 * vmax * np.log(100) / (2 * L)

        for i in range(npml):
            x = (npml - i) / npml
            val = sigma_max * x**2
            sigma[i] = val
            sigma[npts - 1 - i] = val
        return sigma

    sigma_x = pml_sigma(nx, dx)
    sigma_z = pml_sigma(nz, dz)

    rows, cols, data = [], [], []

    for i in range(nz):
        for j in range(nx):

            k = idx(i, j)

            sx = 1.0 + 1j * sigma_x[j] / omega
            sz = 1.0 + 1j * sigma_z[i] / omega

            ax = 1.0 / (sx * dx**2)
            az = 1.0 / (sz * dz**2)

            # 9-point coefficients
            cx = 4.0 / 6.0 * ax
            cz = 4.0 / 6.0 * az
            cdiag = 1.0 / 6.0 * (ax + az) / 2.0

            main = (
                -20.0 / 6.0 * (ax + az) / 2.0
                + omega**2 * m[i, j]
            )

            # Center
            rows.append(k); cols.append(k); data.append(main)

            # Cross neighbors
            for (ii, jj, val) in [
                (i, j-1, cx),
                (i, j+1, cx),
                (i-1, j, cz),
                (i+1, j, cz),
            ]:
                if 0 <= ii < nz and 0 <= jj < nx:
                    rows.append(k)
                    cols.append(idx(ii, jj))
                    data.append(val)

            # Diagonal neighbors
            for (ii, jj) in [
                (i-1, j-1),
                (i-1, j+1),
                (i+1, j-1),
                (i+1, j+1),
            ]:
                if 0 <= ii < nz and 0 <= jj < nx:
                    rows.append(k)
                    cols.append(idx(ii, jj))
                    data.append(cdiag)

    A = sp.coo_matrix((data, (rows, cols)), shape=(n, n))
    return A.tocsr()

In [4]:
import numpy as np
import scipy.sparse as sp

def helmholtzabc(m: np.ndarray,
               omega: float,
               dx: float,
               dz: float,
               npml: int,
               vmax: float) -> sp.csr_matrix:
    """
    2D Helmholtz operator with Clayton–Engquist absorbing boundary condition.

    Parameters
    ----------
    m : (nz, nx) squared slowness (1/v^2)
    omega : angular frequency
    dx, dz : grid spacing
    npml : unused (kept for compatibility)
    vmax : unused (kept for compatibility)

    Returns
    -------
    A : sparse CSR matrix
    """

    nz, nx = m.shape
    n = nx * nz

    def idx(ix, iz):
        return iz * nx + ix

    rows, cols, data = [], [], []

    k_local = omega * np.sqrt(m)

    for iz in range(nz):
        for ix in range(nx):

            row_id = idx(ix, iz)

            # --------------------------------------------------
            # Interior nodes (5-point stencil)
            # --------------------------------------------------
            if 0 < ix < nx-1 and 0 < iz < nz-1:

                main = -2/dx**2 - 2/dz**2 + omega**2 * m[iz, ix]

                rows.append(row_id); cols.append(row_id); data.append(main)

                # Left
                rows.append(row_id); cols.append(idx(ix-1, iz)); data.append(1/dx**2)

                # Right
                rows.append(row_id); cols.append(idx(ix+1, iz)); data.append(1/dx**2)

                # Up
                rows.append(row_id); cols.append(idx(ix, iz-1)); data.append(1/dz**2)

                # Down
                rows.append(row_id); cols.append(idx(ix, iz+1)); data.append(1/dz**2)

            # --------------------------------------------------
            # Clayton–Engquist ABC
            # ∂u/∂n = i k u
            # --------------------------------------------------
            else:

                rows.append(row_id)
                cols.append(row_id)

                # Left boundary
                if ix == 0:
                    data.append(1/dx - 1j * k_local[iz, ix])
                    rows.append(row_id)
                    cols.append(idx(ix+1, iz))
                    data.append(-1/dx)

                # Right boundary
                elif ix == nx-1:
                    data.append(1/dx - 1j * k_local[iz, ix])
                    rows.append(row_id)
                    cols.append(idx(ix-1, iz))
                    data.append(-1/dx)

                # Top boundary
                elif iz == 0:
                    data.append(1/dz - 1j * k_local[iz, ix])
                    rows.append(row_id)
                    cols.append(idx(ix, iz+1))
                    data.append(-1/dz)

                # Bottom boundary
                elif iz == nz-1:
                    data.append(1/dz - 1j * k_local[iz, ix])
                    rows.append(row_id)
                    cols.append(idx(ix, iz-1))
                    data.append(-1/dz)

    A = sp.coo_matrix((data, (rows, cols)), shape=(n, n))
    return A.tocsr()

In [5]:
import numpy as np
import scipy.sparse as sp

def helmholtz_25pt_jo_pml(m, omega, h, npml, vmax):
    """
    2D Helmholtz operator with:
    - True Jo/Operto 25-point stencil
    - Complex-stretch PML
    """

    nz, nx = m.shape
    n = nx * nz

    def idx(i, j):
        return i * nx + j

    # ---------------------------------------------------
    # 1. Build PML stretching profiles
    # ---------------------------------------------------

    def build_s(npts):
        s = np.ones(npts, dtype=complex)

        if npml == 0:
            return s

        sigma_max = 3.0 * vmax * np.log(2) / (2 * npml * h)

        for i in range(npml):
            sigma = sigma_max * ((npml - i) / npml)**2
            s[i] = 1 - 1j * sigma / omega
            s[-(i+1)] = 1 - 1j * sigma / omega

        return s

    sx = build_s(nx)
    sz = build_s(nz)

    # ---------------------------------------------------
    # 2. 25-point stencil weights
    # ---------------------------------------------------

    w = {
        (0, 0): -5.0,
        (0, 1): 4/3, (0, -1): 4/3,
        (1, 0): 4/3, (-1, 0): 4/3,
        (0, 2): -1/12, (0, -2): -1/12,
        (2, 0): -1/12, (-2, 0): -1/12,
        (1, 1): 1/6, (1, -1): 1/6,
        (-1, 1): 1/6, (-1, -1): 1/6,
        (2, 2): -1/48, (2, -2): -1/48,
        (-2, 2): -1/48, (-2, -2): -1/48,
        (1, 2): 1/24, (1, -2): 1/24,
        (-1, 2): 1/24, (-1, -2): 1/24,
        (2, 1): 1/24, (2, -1): 1/24,
        (-2, 1): 1/24, (-2, -1): 1/24,
    }

    rows, cols, data = [], [], []

    # ---------------------------------------------------
    # 3. Assemble operator
    # ---------------------------------------------------

    for i in range(nz):
        for j in range(nx):

            row_id = idx(i, j)

            # Near boundary → simple Dirichlet closure
            if i < 2 or i >= nz-2 or j < 2 or j >= nx-2:
                rows.append(row_id)
                cols.append(row_id)
                data.append(1.0)
                continue

            for (di, dj), coeff in w.items():

                ni = i + di
                nj = j + dj

                # PML scaling
                sx_loc = sx[j] ** abs(dj)
                sz_loc = sz[i] ** abs(di)

                scale = 1.0 / (sx_loc * sz_loc)

                rows.append(row_id)
                cols.append(idx(ni, nj))
                data.append(coeff * scale / h**2)

            # Mass term (no extra scaling needed here)
            rows.append(row_id)
            cols.append(row_id)
            data.append(omega**2 * m[i, j])

    A = sp.coo_matrix((data, (rows, cols)), shape=(n, n))
    return A.tocsr()

In [6]:
import numpy as np
from scipy.special import hankel1

def analytic_2d_helmholtz(nx, nz, dx, dz,
                          omega, velocity,
                          xs, zs):
    """
    Analytic 2D Helmholtz Green's function.

    Parameters
    ----------
    nx, nz : grid size
    dx, dz : spacing
    omega  : angular frequency
    velocity : constant velocity
    xs, zs : source indices

    Returns
    -------
    u : complex ndarray (nz, nx)
    """

    k = omega / velocity

    x = np.arange(nx) * dx
    z = np.arange(nz) * dz

    X, Z = np.meshgrid(x, z)

    x0 = xs * dx
    z0 = zs * dz

    r = np.sqrt((X - x0)**2 + (Z - z0)**2)

    # avoid singularity at source
    r[r == 0] = 1e-10

    u = -1j/4 * hankel1(0, k * r)

    return u

In [7]:
# nx, nz =80, 80
# h=dx = dz = 10
# n = nx * nz
# npml = 10
# f = 7
# omega = 2*np.pi*f

# def idx(i, j):
#     return i * nx + j

# c_true = 2000*np.ones((nz,nx))
# c_true[30:50,30:50] = 2000

# m = 1 / c_true**2
# vmax=2500
# A5 = helmholtz5(m,
#           omega,
#           dx,
#           dz,
#           npml,
#           vmax)
# A9 = helmholtz9(m, omega, dx,dz, npml, vmax)
# A25 =helmholtz_25pt_jo_pml(m, omega, h, npml, vmax)
# Aabc=helmholtzabc(m,
#                omega,
#                dx,
#                dz,
#                npml,
#                vmax)

In [8]:
# # --------------------------------------------------
# # Pressure Wavefield Visualization (Single Shot)
# # --------------------------------------------------

# LU = spla.splu(A25.tocsc())

# # Source position definitions (re-added)
# nsrc = 1
# src_x = np.linspace(40, nx-15, nsrc).astype(int)
# src_z = 10
# sx = src_x[len(src_x)//2]
# sz = src_z

# # Source vector
# q = np.zeros(n, dtype=complex)
# q[idx(sz, sx)] = 1.0

# # Solve
# u = LU.solve(q)
# u2d = u.reshape(nz, nx)

# # --------------------------------------------------
# # Plot
# # --------------------------------------------------

# plt.figure(figsize=(10,6))

# # Receiver x positions
# rec_x = np.arange(10,80,7)
# rec_z = 10

# # -----------------------------
# plt.subplot(2,2,1)
# plt.imshow(np.real(u2d), cmap='RdBu', aspect='auto')
# #plt.scatter(sx, sz, c='yellow', marker='*', s=150, label='Source')
# plt.scatter(src_x, np.ones_like(src_x)*src_z, c='yellow', marker='*', s=150, label='Source')
# plt.scatter(rec_x, np.ones_like(rec_x)*rec_z,
#             c='black', s=10, label='Receivers')
# plt.title("Real Part")
# plt.colorbar()
# plt.legend()

# # -----------------------------
# plt.subplot(2,2,2)
# plt.imshow(np.imag(u2d), cmap='RdBu', aspect='auto')
# plt.scatter(sx, sz, c='yellow', marker='*', s=150)
# plt.title("Imaginary Part")
# plt.colorbar()

# # -----------------------------
# plt.subplot(2,2,3)
# plt.imshow(np.abs(u2d), cmap='viridis', aspect='auto')
# plt.scatter(sx, sz, c='red', marker='*', s=150)
# plt.title("Amplitude |u|")
# plt.colorbar()

# # -----------------------------
# plt.subplot(2,2,4)
# plt.imshow(np.angle(u2d), cmap='twilight', aspect='auto')
# plt.scatter(sx, sz, c='white', marker='*', s=150)
# plt.title("Phase angle(u)")
# plt.colorbar()

# plt.tight_layout()
# plt.show()

In [9]:
# # ==========================================================
# # PRESSURE WAVEFIELD COMPUTATION FOR ALL THREE OPERATORS
# # ==========================================================

# # Solve for A5
# LU5 = spla.splu(A5.tocsc())
# u5 = LU5.solve(q)
# u5_field = u5.reshape(nz, nx)

# # Solve for A9
# LU9 = spla.splu(A9.tocsc())
# u9 = LU9.solve(q)
# u9_field = u9.reshape(nz, nx)

# # Solve for A25
# LU25 = spla.splu(A25.tocsc())
# u25 = LU25.solve(q)
# u25_field = u25.reshape(nz, nx)

# print("Wavefields u5_field, u9_field, and u25_field computed successfully.")

In [10]:
# velocity = 2000
# x_analytic_src = sx
# z_analytic_src = sz
# A = analytic_2d_helmholtz(nx, nz, dx, dz,
#                           omega, velocity,
#                           x_analytic_src, z_analytic_src)

In [11]:
# # Solve for Aabc
# LUabc = spla.splu(Aabc.tocsc())
# uabc = LUabc.solve(q)
# uabc_field = uabc.reshape(nz, nx)

# print("Wavefield uabc_field computed successfully.")

In [12]:
# l2_error_5_vs_analytic = np.linalg.norm(u5_field - A) / np.linalg.norm(A)
# l2_error_9_vs_analytic = np.linalg.norm(u9_field - A) / np.linalg.norm(A)
# l2_error_25_vs_analytic = np.linalg.norm(u25_field - A) / np.linalg.norm(A)
# l2_error_abc_vs_analytic = np.linalg.norm(uabc_field - A) / np.linalg.norm(A)
# print(f"Relative L2 Error (ABC vs Analytic): {l2_error_abc_vs_analytic:.4e}")
# print(f"Relative L2 Error (5-point vs Analytic): {l2_error_5_vs_analytic:.4e}")
# print(f"Relative L2 Error (9-point vs Analytic): {l2_error_9_vs_analytic:.4e}")
# print(f"Relative L2 Error (25-point vs Analytic): {l2_error_25_vs_analytic:.4e}")


In [13]:
# plt.figure(figsize=(12, 6))

# plt.subplot(2, 2, 1)
# plt.imshow(np.real(A), cmap='RdBu', aspect='auto')
# plt.scatter(src_x, np.ones_like(src_x)*src_z, c='yellow', marker='*', s=150, label='Source')
# plt.title("Analytic Solution (Real Part)")
# plt.colorbar()

# plt.subplot(2, 2, 2)
# plt.imshow(np.real(u9_field), cmap='RdBu', aspect='auto')
# plt.scatter(src_x, np.ones_like(src_x)*src_z, c='yellow', marker='*', s=150, label='Source')
# plt.title("9-point Stencil Solution (Real Part)")
# plt.colorbar()

# plt.subplot(2, 2, 3)
# plt.imshow(np.real(u25_field), cmap='RdBu', aspect='auto')
# plt.scatter(src_x, np.ones_like(src_x)*src_z, c='yellow', marker='*', s=150, label='Source')
# plt.title("25-point Stencil Solution (Real Part)")
# plt.colorbar()
# plt.subplot(2, 2, 4)
# plt.imshow(np.real(uabc_field ), cmap='RdBu', aspect='auto')
# plt.scatter(src_x, np.ones_like(src_x)*src_z, c='yellow', marker='*', s=150, label='Source')
# plt.title("5-point Stencil Solution (Real Part)")
# plt.colorbar()

# plt.tight_layout()
# plt.show()